<a href="https://colab.research.google.com/github/Jasonmiller513/Dissertation/blob/Updates/First%20Test%20WA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **STEP 1: CLEAN DATA**

# Import Extensions And Open File

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
import joblib
from collections import defaultdict
df = pd.read_csv('DataCoSupplyChainDataset.csv', encoding='latin1')

# Remove Unnecessary Columns

In [2]:
# List of columns to remove
columns_to_remove = [

    # Identifier columns
    'Customer Id',
    'Category Id',
    'Department Id',

    # Personal/customer detail columns
    'Customer Email',
    'Customer Password',
    'Customer Fname',
    'Customer Lname',
    'Customer Street',
    'Customer Zipcode',
    'Customer Full Name',

    # High-cardinality text columns
    'Product Description',
    'Product Image',
    'Product Name',


    # Duplicate or Redundant Geographic Columns

'Customer Segment',
'Order City',
'Order State',]

# Remove only columns that actually exist in dataframe
existing_columns_to_remove = [
    col for col in columns_to_remove if col in df.columns]

# Drop columns
df = df.drop(columns=existing_columns_to_remove)

# Display remaining columns
print("Remaining Columns:")
print(df.columns.tolist())

# Display shape after removal
print("\nNew Dataset Shape:")
print(df.shape)

Remaining Columns:
['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Name', 'Customer City', 'Customer Country', 'Customer State', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order Country', 'Order Customer Id', 'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order Status', 'Order Zipcode', 'Product Card Id', 'Product Category Id', 'Product Price', 'Product Status', 'shipping date (DateOrders)', 'Shipping Mode']

New Dataset Shape:
(180519, 38)


# Identify missing or erroneous data


In [3]:

print(df.shape)
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0])

(180519, 38)
Order Zipcode    155679
dtype: int64


In [4]:
duplicate_count = df.duplicated().sum()
print(f"Duplicate Rows: {duplicate_count}")

Duplicate Rows: 0


In [5]:
missing = df.isnull().sum()
print(missing[missing > 0])

Order Zipcode    155679
dtype: int64


In [6]:
num_cols = df.select_dtypes(include=['int64','float64']).columns

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())
    cat_cols = df.select_dtypes(include=['object']).columns

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [7]:
non_negative_columns = [
    'Sales',
    'Order Item Quantity',
    'Order Item Total',
    'Product Price',
    'Days for shipping (scheduled)']

for col in non_negative_columns:

    if col in df.columns:

        # Count invalid negatives
        invalid_count = (df[col] < 0).sum()

        print(f"\n{col} Negative Values: {invalid_count}")

        # Replace negatives with median
        median_value = df[df[col] >= 0][col].median()

        df.loc[df[col] < 0, col] = median_value


Sales Negative Values: 0

Order Item Quantity Negative Values: 0

Order Item Total Negative Values: 0

Product Price Negative Values: 0


In [8]:
numeric_cols = df.select_dtypes(include=np.number).columns

outlier_summary = {}

for col in numeric_cols:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = ((df[col] < lower) | (df[col] > upper)).sum()

    outlier_summary[col] = outliers

outlier_df = pd.DataFrame.from_dict(
    outlier_summary,
    orient='index',
    columns=['Outlier Count']
)

print(outlier_df.sort_values('Outlier Count', ascending=False))

                               Outlier Count
Order Zipcode                          24818
Benefit per order                      18942
Order Profit Per Order                 18942
Order Item Profit Ratio                17300
Order Item Discount                     7537
Order Item Product Price                2048
Product Price                           2048
Sales per customer                      1943
Order Item Total                        1943
Longitude                               1414
Order Customer Id                       1198
Sales                                    488
Latitude                                   9
Late_delivery_risk                         0
Days for shipment (scheduled)              0
Days for shipping (real)                   0
Order Item Quantity                        0
Order Item Id                              0
Order Item Cardprod Id                     0
Order Item Discount Rate                   0
Order Id                                   0
Product Ca

# Winsorize Outliers

In [9]:
for col in numeric_cols:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = np.where(df[col] < lower, lower, df[col])
    df[col] = np.where(df[col] > upper, upper, df[col])

# Log Transform Highly Skewed Variables

In [10]:
skew_cols = [
    'Order Item Total',
    'Order Profit Per Order',
    'Shipping Cost',
    'Profit Per Unit'
]

for col in skew_cols:
    if col in df.columns:
        df[col] = np.log1p(df[col] - df[col].min())

In [11]:
print("\nRemaining Missing Values:")
print(df.isnull().sum().sum())

print("\nFinal Dataset Shape:")
print(df.shape)

print("\nDataset Preview:")
print(df.head())


Remaining Missing Values:
0

Final Dataset Shape:
(180519, 38)

Dataset Preview:
       Type  Days for shipping (real)  Days for shipment (scheduled)  \
0     DEBIT                       3.0                            4.0   
1  TRANSFER                       5.0                            4.0   
2      CASH                       4.0                            4.0   
3     DEBIT                       3.0                            4.0   
4   PAYMENT                       2.0                            4.0   

   Benefit per order  Sales per customer   Delivery Status  \
0          91.250000          314.640015  Advance shipping   
1         -79.700005          311.359985     Late delivery   
2         -79.700005          309.720001  Shipping on time   
3          22.860001          304.809998  Advance shipping   
4         134.210007          298.250000  Advance shipping   

   Late_delivery_risk   Category Name Customer City Customer Country  ...  \
0                 0.0  Sporting Goo

# Remove Constant and Near-Constant Features

In [12]:
constant_cols = [
    col for col in df.columns
    if df[col].nunique() == 1
]

print("Constant Features:")
print(constant_cols)

df.drop(columns=constant_cols, inplace=True)
near_constant_cols = []

for col in df.columns:

    freq = df[col].value_counts(normalize=True)

    # Check if freq is empty (e.g., column was entirely NaN) or if a single value dominates
    if freq.empty or freq.iloc[0] > 0.99:
        near_constant_cols.append(col)

print("Near Constant Features:")
print(near_constant_cols)

df.drop(columns=near_constant_cols, inplace=True)

Constant Features:
['Order Zipcode', 'Product Status']
Near Constant Features:
[]


# Remove Duplicate Rows

In [13]:
print("Rows before:", len(df))

df.drop_duplicates(inplace=True)

print("Rows after:", len(df))

Rows before: 180519
Rows after: 180519


# Verify Dataset Quality

In [14]:
print("Remaining Missing Values:")
print(df.isnull().sum().sum())

print("Dataset Shape:")
print(df.shape)

Remaining Missing Values:
0
Dataset Shape:
(180519, 36)


# Recompute Numeric Columns

In [15]:
numeric_cols = df.select_dtypes(include=['int64','float64']).columns.tolist()

print(f"Number of numeric features: {len(numeric_cols)}")
print(numeric_cols)

Number of numeric features: 22
['Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Late_delivery_risk', 'Latitude', 'Longitude', 'Order Customer Id', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Product Card Id', 'Product Category Id', 'Product Price']


#Save Data

In [16]:
# Save cleaned RL dataset to CSV in Google Colab

output_file = 'dataco_rl_cleaned.csv'

# Save dataframe
df.to_csv(output_file, index=False)

print(f"Dataset saved as: {output_file}")

Dataset saved as: dataco_rl_cleaned.csv


# **STEP 2: FEATURE ENGINEERING**

In [17]:
df = pd.read_csv('dataco_rl_cleaned.csv', encoding='latin1')

print("Dataset Loaded")
print(df.shape)

Dataset Loaded
(180519, 36)


# Date and Time Features

In [18]:
# Convert order date to datetime
df['order date (DateOrders)'] = pd.to_datetime(
    df['order date (DateOrders)'],
    errors='coerce')

# Extract temporal features
df['order_year'] = df['order date (DateOrders)'].dt.year
df['order_month'] = df['order date (DateOrders)'].dt.month
df['order_day'] = df['order date (DateOrders)'].dt.day
df['order_dayofweek'] = df['order date (DateOrders)'].dt.dayofweek
df['is_weekend'] = df['order_dayofweek'].isin([5,6]).astype(int)


# Create a Product-Region Matrix and Network

In [19]:
# Create a Product-Region Matrix
product_region = (
    df.groupby("Product Category Id")["Order Region"]
      .nunique()
      .reset_index(name="num_regions")
      .sort_values("num_regions", ascending=False))

In [20]:
order_features = (
    df.groupby("Order Id")
      .agg(
          unique_products=("Product Category Id", "nunique"),
          total_quantity=("Order Item Quantity", "sum"),
          total_sales=("Sales", "sum"))
      .reset_index())

order_features["complex_order"] = (
    order_features["unique_products"] >= 3
).astype(int)

In [21]:
network = (
    df.groupby(["Product Category Id", "Order Region"])
      .agg(
          orders=("Order Id", "nunique"),
          quantity=("Order Item Quantity", "sum"),
          revenue=("Sales", "sum"))
      .reset_index())

# Explore Product Statistics

In [22]:
product_stats = (
    network.groupby("Product Category Id")
           .agg(
               regions_served=("Order Region", "nunique"),
               total_orders=("orders", "sum"),
               total_quantity=("quantity", "sum"),
               total_revenue=("revenue", "sum"))
           .reset_index())


# Create Shipping Mode Statistics

In [23]:
import pandas as pd
import numpy as np

# Ensure 'Shipping Mode_str' and 'Order Region_str' are available
df['Shipping Mode_str'] = df['Shipping Mode']
df['Order Region_str'] = df['Order Region']

df = df[df['Order Region_str'] == 'West Africa'].copy()

# Historical shipping mode performance
mode_stats = (
    df.groupby('Shipping Mode_str')
      .agg({
          'Order Profit Per Order':'mean',
          'Late_delivery_risk':'mean',
          'Days for shipping (real)':'mean'
      })
      .reset_index()
)

print(mode_stats)

  Shipping Mode_str  Order Profit Per Order  Late_delivery_risk  \
0       First Class                4.368462            0.978402   
1          Same Day                4.236680            0.354260   
2      Second Class                4.310360            0.772358   
3    Standard Class                4.396597            0.374560   

   Days for shipping (real)  
0                  2.000000  
1                  0.390135  
2                  3.979675  
3                  4.010563  


# Define Multi-Region Stock

In [24]:
region_threshold = product_stats["regions_served"].median()
order_threshold = product_stats["total_orders"].median()

product_stats["likely_multi_region_stocked"] = (
    (product_stats["regions_served"] >= region_threshold)
    &
    (product_stats["total_orders"] >= order_threshold)
).astype(int)

#Create a stock score

In [25]:
product_stats["stocking_score"] = (
    0.5 *
    (
        product_stats["regions_served"]
        / product_stats["regions_served"].max())
    +
    0.5 *
    (
        product_stats["total_orders"]
        / product_stats["total_orders"].max()))


# Merge the network

In [26]:
network = network.merge(
    product_stats[
        [
            "Product Category Id",
            "regions_served",
            "stocking_score",
            "likely_multi_region_stocked"
        ]
    ],
    on="Product Category Id",
    how="left")

# Find Edge Weight

In [27]:
network["edge_weight"] = (
    0.4 *
    (network["orders"] / network["orders"].max())
    +
    0.3 *
    (network["quantity"] / network["quantity"].max())
    +
    0.3 *
    (network["revenue"] / network["revenue"].max()))


 # Find candidate shipping regions

In [28]:
candidate_regions = (
    network.sort_values(
        ["Product Category Id", "edge_weight"],
        ascending=[True, False])
    .groupby("Product Category Id")
    .head(5))

# Save Outputs

In [29]:
network.to_csv(
    "product_region_network.csv",
    index=False
)

candidate_regions.to_csv(
    "candidate_fulfillment_regions.csv",
    index=False
)

product_stats.to_csv(
    "product_stocking_scores.csv",
    index=False
)

print("Files created successfully")

Files created successfully


# Shipping Engineering

In [30]:
df['order date (DateOrders)'] = pd.to_datetime(
    df['order date (DateOrders)'],
    errors='coerce'
)
df['shipping date (DateOrders)'] = pd.to_datetime(
    df['shipping date (DateOrders)'],
    errors='coerce'
)

# Actual shipping time

df['actual_ship_days'] = (
    df['shipping date (DateOrders)']
    - df['order date (DateOrders)']
).dt.days

# Difference from scheduled

df['shipping_delay'] = (
    df['Days for shipping (real)']
    - df['Days for shipment (scheduled)']
)

# Binary on-time flag

df['on_time_delivery'] = (
    df['shipping_delay'] <= 0
).astype(int)

# Late shipment flag

df['late_delivery'] = (
    df['shipping_delay'] > 0
).astype(int)

# Profit Engineering

In [31]:
# Margin percentage

df['margin_pct'] = (
    df['Order Profit Per Order']
    / df['Sales']
)

# Profit per item

df['profit_per_item'] = (
    df['Order Profit Per Order']
    / df['Order Item Quantity']
)

# Discount percentage

df['discount_pct'] = (
    df['Order Item Discount']
    / df['Order Item Product Price']
)

# Product Demand Features

In [32]:
product_stats = (
    df.groupby('Product Category Id')
      .agg(
          product_orders=('Order Id','nunique'),
          product_quantity=('Order Item Quantity','sum'),
          product_revenue=('Sales','sum')
      )
      .reset_index()
)

df = df.merge(
    product_stats,
    on='Product Category Id',
    how='left'
)

# Regional Features

In [33]:
region_stats = (
    df.groupby('Order Region')
      .agg(
          region_orders=('Order Id','nunique'),
          region_sales=('Sales','sum'),
          region_profit=('Order Profit Per Order','sum')
      )
      .reset_index()
)

df = df.merge(
    region_stats,
    on='Order Region',
    how='left'
)

# Fulfillment Network Features

In [34]:
stocking = pd.read_csv(
    "product_stocking_scores.csv"
)

df = df.merge(
    stocking[
        [
            'Product Category Id',
            'stocking_score',
            'regions_served',
            'likely_multi_region_stocked'
        ]
    ],
    on='Product Category Id',
    how='left'
)

# Shipping Mode Features

In [35]:
speed_mapping = {
    'Same Day':4,
    'First Class':3,
    'Second Class':2,
    'Standard Class':1
}
df['shipping_mode_speed'] = df['Shipping Mode'].map(speed_mapping)

# Create string versions of Shipping Mode and Order Region for later use
df['Shipping Mode_str'] = df['Shipping Mode']
df['Order Region_str'] = df['Order Region']

action_id_mapping = {
    'Standard Class': 0,    # Maps to key 0 in shipping_cost_map
    'Second Class': 1,     # Maps to key 1 in shipping_cost_map
    'First Class': 2,      # Maps to key 2 in shipping_cost_map
    'Same Day': 3          # Maps to key 3 in shipping_cost_map
}
df['action_id'] = df['Shipping Mode'].map(action_id_mapping)

shipping_cost_map = {
    0: 1.0,    # Standard
    1: 1.5,    # Second
    2: 2.0,    # First
    3: 3.0     # Same Day
}

BASE_COST = 10

df['Mode_Cost'] = (
    BASE_COST *
    df['action_id'].map(shipping_cost_map)
)

# Order Complexity Features

In [36]:
order_complexity = (
    df.groupby('Order Id')
      .agg(
          unique_products=('Product Category Id','nunique'),
          total_quantity=('Order Item Quantity','sum'),
          total_sales=('Sales','sum')
      )
      .reset_index()
)

df = df.merge(
    order_complexity,
    on='Order Id',
    how='left'
)

df['complex_order'] = (
    df['unique_products'] >= 3
).astype(int)

# Route-Based Shipping Costs

In [37]:
df = pd.read_csv('dataco_rl_feature_engineered.csv')

# Ensure 'Shipping Cost' is present, assuming it should align with 'Mode_Cost'
df['Shipping Cost'] = df['Mode_Cost']

route_cost = (
    df.groupby(
        ['Order Region','Shipping Mode']
    )['Shipping Cost']
    .mean()
    .reset_index()
)

route_cost.rename(
    columns={
        'Shipping Cost':'route_cost'
    },
    inplace=True
)

df = df.merge(
    route_cost,
    on=['Order Region','Shipping Mode'],
    how='left'
)

# RL State Features

In [38]:
state_features = [
    'Product Category Id',
    'Order Region',
    'Market',
    'Shipping Mode',
    'stocking_score',
    'regions_served',
    'likely_multi_region_stocked',
    'unique_products',
    'total_quantity',
    'shipping_delay',
    'margin_pct',
    'profit_per_item',
    'order_month',
    'order_dayofweek'
]

# Reward Features

In [39]:
df['reward'] = (
    df['Order Profit Per Order']
    + 100 * df['on_time_delivery']
    - 20 * df['late_delivery']
)
LATE_PENALTY = 10

df['reward'] = (
    df['Order Profit Per Order']
    - df['Mode_Cost']
    - LATE_PENALTY * df['Late_delivery_risk']
)
df['net_profit'] = (
    df['Order Profit Per Order']
    - df['Shipping Cost']
)

# Save Engineered Dataset

In [40]:
df.to_csv(
    "dataco_rl_feature_engineered.csv",
    index=False
)

print(df.shape)

(3696, 71)


# **STEP 3: ENCODING CATEGORICAL VARIABLES**

In [41]:

df = pd.read_csv('dataco_rl_feature_engineered.csv', encoding='latin1')


print("Dataset Loaded")
print(df.shape)

Dataset Loaded
(3696, 71)


In [42]:
one_hot_cols = [
    'Shipping Mode',
    'Market',
    'Order Region'
]

df = pd.get_dummies(
    df,
    columns=one_hot_cols,
    drop_first=False
)

df.to_csv(
    "dataco_rl_onehot.csv",
    index=False
)

print(df.shape)

(3696, 74)


# Filter on West Africa

In [43]:
df = pd.read_csv("dataco_rl_onehot.csv")

# Keep only West Africa orders
west_africa_df = df[df["Order Region_West Africa"] == 1].copy()

print("Number of West Africa orders:", len(west_africa_df))

Number of West Africa orders: 3696


# **STEP 4: TRAIN/TEST TEMPORAL SPLIT**

In [44]:
import pandas as pd

df = pd.read_csv("dataco_rl_onehot.csv")

df = df[df["Order Region_West Africa"] == 1].copy()

df['order date (DateOrders)'] = pd.to_datetime(
    df['order date (DateOrders)']
)

df = df.sort_values(
    'order date (DateOrders)'
)

n = len(df)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df.iloc[:train_end]
val_df = df.iloc[train_end:val_end]
test_df = df.iloc[val_end:]

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(2587, 74)
(554, 74)
(555, 74)


In [45]:
# Save datasets

train_df.to_csv(
    "dataco_rl_train.csv",
    index=False
)

val_df.to_csv(
    "dataco_rl_validation.csv",
    index=False
)

test_df.to_csv(
    "dataco_rl_test.csv",
    index=False
)

print("Files saved successfully")

Files saved successfully


# STEP 5: STATE-SPACE DISCRETIZATION

# Feature Selection

In [46]:
state_features = [
    'Profit Margin',
    'Shipping Cost Ratio',
    'Historical On-Time Rate',
    'Regional Delay Score',
    'Product Category Id'
]

# Fit Bins Using Training Data Only

In [47]:
train_df = pd.read_csv("dataco_rl_train.csv")

# Use train_df instead of df for binning operations
train_df['shipping_cost_bin'] = pd.qcut(
    train_df['Shipping Cost'],
    q=5,
    labels=False,
    duplicates='drop'
)

train_df['order_value_bin'] = pd.qcut(
    train_df['Order Item Total'],
    q=5,
    labels=False,
    duplicates='drop'
)

# The 'Order Region' column does not exist in train_df.
# Since train_df is already filtered to 'West Africa' and 'Order Region'
# was one-hot encoded, 'Order Region_bin' will be a constant.
# Assigning a constant value to create the expected column.
train_df['Order Region_bin'] = 0

exclude_cols = [
    'Reward',
    'Late_delivery_risk',
    'Action_ID'
]

continuous_features = [
    col for col in train_df.select_dtypes(include='number').columns
    if col not in exclude_cols
    and not col.endswith('_bin') # Exclude the newly created bin columns from being binned again
]

bin_edges = {}

# Create 10 bins for each feature
for col in continuous_features:
    # Handle cases where all values are the same to prevent qcut errors
    if train_df[col].nunique() == 1:
        # If all values are the same, create a single bin from min to min (or min to min + epsilon)
        min_val = train_df[col].min()
        bins = np.array([min_val, min_val + 1e-9]) # Add a small epsilon to ensure two distinct points
    else:
        _, bins = pd.qcut(
            train_df[col],
            q=10,
            labels=False,
            retbins=True,
            duplicates='drop'
        )
    bin_edges[col] = bins

# Save bin definitions
joblib.dump(
    bin_edges,
    "state_bins.pkl"
)

print("Bins created successfully")

Bins created successfully


In [48]:
bin_counts = {}

for col in continuous_features:
    _, bins = pd.qcut(
        train_df[col],
        q=10,
        retbins=True,
        duplicates='drop'
    )

    bin_counts[col] = len(bins) - 1

print(bin_counts)

{'Days for shipping (real)': 5, 'Days for shipment (scheduled)': 3, 'Benefit per order': 10, 'Sales per customer': 10, 'Latitude': 10, 'Longitude': 10, 'Order Customer Id': 10, 'Order Id': 10, 'Order Item Cardprod Id': 9, 'Order Item Discount': 10, 'Order Item Discount Rate': 10, 'Order Item Id': 10, 'Order Item Product Price': 8, 'Order Item Profit Ratio': 10, 'Order Item Quantity': 4, 'Sales': 9, 'Order Item Total': 10, 'Order Profit Per Order': 10, 'Product Card Id': 9, 'Product Category Id': 9, 'Product Price': 8, 'order_year': 0, 'order_month': 4, 'order_day': 10, 'order_dayofweek': 6, 'is_weekend': 1, 'shipping_mode_speed': 3, 'action_id': 3, 'Mode_Cost': 3, 'actual_ship_days': 5, 'shipping_delay': 5, 'on_time_delivery': 1, 'late_delivery': 1, 'margin_pct': 10, 'profit_per_item': 10, 'discount_pct': 10, 'product_orders': 8, 'product_quantity': 9, 'product_revenue': 8, 'region_orders': 0, 'region_sales': 0, 'region_profit': 0, 'stocking_score': 8, 'regions_served': 1, 'likely_mult

# Apply Bins to Train/Test Data

In [49]:
train_df = pd.read_csv("dataco_rl_train.csv")
test_df = pd.read_csv("dataco_rl_test.csv")

bin_edges = joblib.load("state_bins.pkl")

continuous_features = list(bin_edges.keys())

for col in continuous_features:

    train_df[col + "_bin"] = np.digitize(
        train_df[col],
        bin_edges[col][1:-1]
    )

    test_df[col + "_bin"] = np.digitize(
        test_df[col],
        bin_edges[col][1:-1]
    )

train_df.to_csv(
    "dataco_rl_train_discrete.csv",
    index=False
)

test_df.to_csv(
    "dataco_rl_test_discrete.csv",
    index=False
)

print("Discretization complete")

Discretization complete


# Build RL States

In [50]:
import pandas as pd
import numpy as np

# Load the discretized training data
df = pd.read_csv("dataco_rl_train_discrete.csv")

# All data is already filtered for 'West Africa', so 'Order Region_bin' is a constant
df['Order Region_bin'] = 0

state_cols = [
    'Product Category Id_bin',
    'stocking_score_bin',
    'margin_pct_bin',
    'shipping_delay_bin',
    'Order Region_bin',
    'Order Item Total_bin', # Corrected name from 'order_value_bin'
    'Shipping Cost_bin'     # Corrected name from 'shipping_cost_bin'
]

df['state'] = (
    df[state_cols]
      .astype(str)
      .agg('_'.join, axis=1)
)

state_map = {
    s:i for i,s in enumerate(df['state'].unique())
}

df['state_id'] = df['state'].map(state_map)

num_states = len(state_map)

print("States:", num_states)

States: 1011


# Create State IDs

In [51]:
train_df = pd.read_csv("dataco_rl_train_discrete.csv")

state_cols = [
    'Product Category Id_bin',
    'stocking_score_bin',
    'margin_pct_bin',
    'shipping_delay_bin'
]

train_df['state'] = (
    train_df[state_cols]
      .astype(str)
      .agg('_'.join, axis=1)
)

state_encoder = LabelEncoder()

train_df['state_id'] = state_encoder.fit_transform(
    train_df['state']
)

joblib.dump(
    state_encoder,
    "state_encoder.pkl"
)

train_df['state_id'] = (
    train_df['state_id']
    .astype(int)
)

train_df['action_id'] = (
    train_df['action_id']
    .astype(int)
)

print(
    "Unique states:",
    train_df['state_id'].nunique()
)

Unique states: 377


# Create Action IDs

In [53]:
candidate_regions = pd.read_csv(
    "candidate_fulfillment_regions.csv"
)

shipping_modes = [
    "Standard Class",
    "Second Class",
    "First Class",
    "Same Day"
]


action_map = {
    a:i for i,a in enumerate(shipping_modes)
}

num_actions = len(shipping_modes)

# Reconstruct 'Shipping Mode' from 'shipping_mode_speed'
# This mapping is derived from previous cells in the notebook.
shipping_mode_map_reverse = {
    1: 'Standard Class',
    2: 'Second Class',
    3: 'First Class',
    4: 'Same Day'
}
train_df['Shipping Mode'] = train_df['shipping_mode_speed'].map(shipping_mode_map_reverse)

# Reconstruct 'Order Region'. Since the data was filtered for 'West Africa' earlier,
# all orders in train_df belong to 'West Africa'.
train_df['Order Region'] = 'West Africa'

train_df['route_action'] = (
    train_df['Order Region'].astype(str)
    + '_'
    + train_df['Shipping Mode'].astype(str)
)

train_df['action_id'] = pd.factorize(
    train_df['route_action']
)[0]

action_lookup = dict(
    enumerate(
        train_df['route_action']
        .astype('category')
        .cat.categories
    )
)

# Build Action Lookup Table

In [55]:
action_lookup = {}

for mode in shipping_modes:

    subset = train_df[train_df['Shipping Mode_str'] == mode]

    action_lookup[mode] = {
        'profit':
            subset['Order Profit Per Order'].mean(),

        'late_rate':
            subset['Late_delivery_risk'].mean(),

        'ship_days':
            subset['Days for shipping (real)'].mean()
    }

action_lookup

{'Standard Class': {'profit': np.float64(4.421881657901078),
  'late_rate': np.float64(0.3526048284625159),
  'ship_days': np.float64(3.9275730622617533)},
 'Second Class': {'profit': np.float64(4.298993400059995),
  'late_rate': np.float64(0.7660455486542443),
  'ship_days': np.float64(3.9523809523809526)},
 'First Class': {'profit': np.float64(4.435445490278215),
  'late_rate': np.float64(0.9776536312849162),
  'ship_days': np.float64(2.0)},
 'Same Day': {'profit': np.float64(4.1937452454247905),
  'late_rate': np.float64(0.38372093023255816),
  'ship_days': np.float64(0.38953488372093026)}}

In [56]:
action_encoder = LabelEncoder()

# Reconstruct 'Shipping Mode' string from 'shipping_mode_speed'
shipping_mode_map_reverse = {
    1: 'Standard Class',
    2: 'Second Class',
    3: 'First Class',
    4: 'Same Day'
}
train_df['Shipping Mode_str'] = train_df['shipping_mode_speed'].map(shipping_mode_map_reverse)

# Reconstruct 'Order Region' string from one-hot encoded columns
# Exclude the 'Order Region_str' column itself from the list of one-hot encoded columns
order_region_cols = [col for col in train_df.columns if col.startswith('Order Region_') and not col.endswith('_str')]
# Use idxmax to find the column with value 1, then strip the prefix
train_df['Order Region_str'] = train_df[order_region_cols].idxmax(axis=1).str.replace('Order Region_', '')

# Create the 'action' column by combining the reconstructed strings
train_df['action'] = train_df['Order Region_str'] + '|' + train_df['Shipping Mode_str']

# Now fit and transform the action column
train_df['action_id'] = action_encoder.fit_transform(
    train_df['action']
)

joblib.dump(
    action_encoder,
    "action_encoder.pkl"
)

print(
    "Unique actions:",
    train_df['action_id'].nunique()
)

Unique actions: 4


# Check State Space Size

In [57]:
print(
    "Unique states:",
    train_df['state'].nunique()
)

print(
    "Unique actions:",
    train_df['action'].nunique()
)

print(
    "Q-table size:",
    train_df['state'].nunique()
    *
    train_df['action'].nunique()
)

Unique states: 377
Unique actions: 4
Q-table size: 1508


# **STEP 6: Q-LEARNING IMPLEMENTATION**


In [58]:
train_df = pd.read_csv("dataco_rl_train_discrete.csv")

# All data is already filtered for 'West Africa', so 'Order Region_bin' is a constant
train_df['Order Region_bin'] = 0

# Encode states/actions

from sklearn.preprocessing import LabelEncoder

state_encoder = LabelEncoder()
action_encoder = LabelEncoder()

state_columns = [
    'Product Category Id_bin',
    'stocking_score_bin',
    'margin_pct_bin',
    'shipping_delay_bin',
    'Order Region_bin',
    'Shipping Cost_bin', # Corrected name from 'shipping_cost_bin'
    'Order Item Total_bin' # Corrected name from 'order_value_bin'
]


train_df['state'] = train_df[state_columns] \
    .astype(str) \
    .agg('|'.join, axis=1)

train_df['state_id'] = state_encoder.fit_transform(
    train_df['state']
)

# Recreate 'action' column
shipping_mode_map_reverse = {
    1: 'Standard Class',
    2: 'Second Class',
    3: 'First Class',
    4: 'Same Day'
}
train_df['Shipping Mode_str'] = train_df['shipping_mode_speed'].map(shipping_mode_map_reverse)

order_region_cols = [col for col in train_df.columns if col.startswith('Order Region_') and not col.endswith('_str')]
train_df['Order Region_str'] = train_df[order_region_cols].idxmax(axis=1).str.replace('Order Region_', '')

train_df['action'] = train_df['Order Region_str'] + '|' + train_df['Shipping Mode_str']

train_df['action_id'] = action_encoder.fit_transform(
    train_df['action']
)


# Q-Learning Parameters


alpha = 0.10      # learning rate
gamma = 0.95      # discount factor
epsilon = 0.20    # exploration rate

episodes = 50

n_states = train_df['state_id'].nunique()
n_actions = train_df['action_id'].nunique()

# Historical Route Profit Feature

In [59]:
route_profit = (
    train_df.groupby(
        ['Order Region_str','Shipping Mode_str'] # Use the _str versions created in m2F8OIGd2x7p
    )['Order Profit Per Order']
    .mean()
    .reset_index()
)

route_profit.rename(
    columns={
        'Order Profit Per Order':'historical_route_profit'
    },
    inplace=True
)

train_df = train_df.merge(
    route_profit,
    left_on=['Order Region_str','Shipping Mode_str'], # Merge on _str versions
    right_on=['Order Region_str','Shipping Mode_str'],
    how='left'
)

# Dual-Goal Reward Fuction

In [94]:
PROFIT_WEIGHT = 3.0
LATE_PENALTY = 2.0
TIME_PENALTY = 0.25

def calculate_reward(row):
    profit = row["Order Profit Per Order"]
    late = row["Late_delivery_risk"]
    ship_time = row["Days for shipping (real)"]

    reward = (
        PROFIT_WEIGHT * profit
        - LATE_PENALTY * late
        - TIME_PENALTY * ship_time
    )

    return reward

# Initialize Q-Table

In [95]:
Q = np.zeros((n_states, n_actions))

alpha = 0.1
gamma = 0.95

epsilon = 1.0
epsilon_min = 0.05
epsilon_decay = 0.995

epsilon_start = 1.0
epsilon_min = 0.05
epsilon_decay = 0.995

epsilon = epsilon_start

episodes = 500

epsilon = max(
    epsilon_min,
    epsilon * epsilon_decay
)

In [96]:
print("Rows:", len(train_df))
print("States:", train_df['state_id'].nunique())
print("Actions:", train_df['action_id'].nunique())

Rows: 2587
States: 1011
Actions: 4


In [97]:
state_lookup = {
    state: group
    for state, group in train_df.groupby('state_id')
}

In [116]:
profit_by_mode = train_df.groupby("Shipping Mode_str")[
    "Order Profit Per Order"
].mean()

print(profit_by_mode)

Shipping Mode_str
First Class       4.435445
Same Day          4.193745
Second Class      4.298993
Standard Class    4.421882
Name: Order Profit Per Order, dtype: float64


In [117]:
profit_by_mode

,Order Profit Per Order
Shipping Mode_str,
First Class,4.435445
Same Day,4.193745
Second Class,4.298993
Standard Class,4.421882


In [118]:
train_df["reward"].describe()

,reward
count,2587.000000
mean,11.262662
std,4.077911
min,-3.500000
25%,10.779169
50%,12.243915
75%,13.494757
max,16.342797


In [119]:
policy = np.argmax(Q, axis=1)

policy_actions = action_encoder.inverse_transform(policy)

pd.Series(policy_actions).value_counts(normalize=True) * 100

,proportion
West Africa|First Class,31.552918
West Africa|Same Day,27.101879
West Africa|Second Class,24.530168
West Africa|Standard Class,16.815035


In [120]:
policy = np.argmax(Q, axis=1)

policy_actions = action_encoder.inverse_transform(policy)

print(
    pd.Series(policy_actions)
      .value_counts(normalize=True)
      .mul(100)
)

West Africa|First Class       31.552918
West Africa|Same Day          27.101879
West Africa|Second Class      24.530168
West Africa|Standard Class    16.815035
Name: proportion, dtype: float64


In [135]:
# Q-Learning Training

import numpy as np
import pandas as pd

# Recalculate reward if needed
train_df["reward"] = train_df.apply(calculate_reward, axis=1)

n_states = train_df["state_id"].nunique()
n_actions = train_df["action_id"].nunique()

Q = np.zeros((n_states, n_actions))

alpha = 0.1
gamma = 0.50      # lower than 0.95 for offline tabular data
epsilon = 1.0
epsilon_min = 0.05
epsilon_decay = 0.995
episodes = 100

reward_table = (
    train_df
    .groupby(["state_id", "action_id"])["reward"]
    .mean()
    .to_dict()
)

state_ids = train_df["state_id"].to_numpy()
mean_reward = train_df["reward"].mean()

reward_history = []

for episode in range(episodes):
    total_reward = 0

    for i in range(len(state_ids) - 1):
        state = int(state_ids[i])
        next_state = int(state_ids[i + 1])

        if np.random.rand() < epsilon:
            action = np.random.randint(n_actions)
        else:
            action = np.argmax(Q[state])

        reward = reward_table.get((state, action), mean_reward)

        Q[state, action] += alpha * (
            reward + gamma * np.max(Q[next_state]) - Q[state, action]
        )

        total_reward += reward

    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    reward_history.append(total_reward)

    if episode % 10 == 0:
        print(f"Episode {episode}: Reward = {total_reward:.2f}")

print("Training complete")
print("Q min:", Q.min())
print("Q max:", Q.max())
print("Q mean:", Q.mean())

Episode 0: Reward = 29056.13
Episode 10: Reward = 29302.72
Episode 20: Reward = 29369.28
Episode 30: Reward = 29373.36
Episode 40: Reward = 29426.97
Episode 50: Reward = 29721.41
Episode 60: Reward = 29805.88
Episode 70: Reward = 30004.88
Episode 80: Reward = 30027.47
Episode 90: Reward = 30062.22
Training complete
Q min: 4.256329268561609
Q max: 28.96652144542605
Q mean: 21.132698262875905


In [136]:
policy = np.argmax(Q, axis=1)
policy_actions = action_encoder.inverse_transform(policy)

pd.Series(policy_actions).value_counts(normalize=True) * 100

,proportion
West Africa|Standard Class,34.718101
West Africa|Same Day,23.343225
West Africa|First Class,21.266073
West Africa|Second Class,20.672601


# Extract the Learned Policy

In [137]:
best_action_id = np.argmax(Q[state])

# Map the action ID back to the shipping mode name
recommended_shipping_mode = shipping_modes[best_action_id]

# Now use the shipping mode name to get the details from action_lookup
recommended_route_details = action_lookup[recommended_shipping_mode]

print(recommended_route_details)

{'profit': np.float64(4.298993400059995), 'late_rate': np.float64(0.7660455486542443), 'ship_days': np.float64(3.9523809523809526)}


In [138]:
policy_df = pd.DataFrame({
    'state_id': np.arange(n_states),
    'best_action': np.argmax(Q, axis=1) # Get best action for each state
})

policy_df['best_shipping_mode'] = (
    policy_df['best_action']
      .map(dict(enumerate(shipping_modes))) # Use shipping_modes instead of actions
)

policy_df.head()

,state_id,best_action,best_shipping_mode
0,0,1,Second Class
1,1,2,First Class
2,2,2,First Class
3,3,0,Standard Class
4,4,1,Second Class


# Convert Best Actions Back to Business Decisions

In [139]:
best_actions = []

for state in range(n_states):

    action_id = np.argmax(Q[state])

    action_name = action_encoder.inverse_transform(
        [action_id]
    )[0]

    best_actions.append(
        [state, action_name]
    )

policy_df = pd.DataFrame(
    best_actions,
    columns=["state_id", "recommended_action"]
)

policy_df.head()

,state_id,recommended_action
0,0,West Africa|Same Day
1,1,West Africa|Second Class
2,2,West Africa|Second Class
3,3,West Africa|First Class
4,4,West Africa|Same Day


# Evaluate Recommended Shipping Modes

In [140]:
recommended_modes = []

for state in train_df['state_id']:

    action = np.argmax(Q[state])

    recommended_modes.append(shipping_modes[action])

df['recommended_mode'] = recommended_modes

print(
    df['recommended_mode']
      .value_counts()
)

recommended_mode
Same Day          1115
Standard Class     502
First Class        501
Second Class       469
Name: count, dtype: int64


# Measure Improvement

In [141]:
train_df["recommended_action"] = train_df["state_id"].apply(
    lambda s: np.argmax(Q[int(s)])
)

baseline_profit = train_df["Order Profit Per Order"].mean()

recommended_profit = (
    train_df
    .groupby("recommended_action")["Order Profit Per Order"]
    .mean()
    .mean()
)

profit_improvement = (
    (recommended_profit - baseline_profit) / abs(baseline_profit)
) * 100

print("Baseline Avg Profit:", baseline_profit)
print("Recommended Avg Profit:", recommended_profit)
print("Profit Improvement %:", profit_improvement)

print("Historical Days:", historical_days)
print("RL Days:", rl_days)

print(
    "Profit Improvement:",
    round(
        (rl_profit-historical_profit)
        /
        abs(historical_profit)
        *100,
        2
    ),
    "%"
)

print(
    "Shipping Time Improvement:",
    round(
        (historical_days-rl_days)
        /
        historical_days
        *100,
        2
    ),
    "%"
)

Baseline Avg Profit: 4.385647162542691
Recommended Avg Profit: 4.290375350909607
Profit Improvement %: -2.1723546856844607
Historical Days: 3.4302280633938924
RL Days: 2.1473725567960447
Profit Improvement: -1.69 %
Shipping Time Improvement: 37.4 %


In [129]:
# Historical profit by mode
profit_by_mode = train_df.groupby(
    "Shipping Mode_str"
)["Order Profit Per Order"].mean()

print(profit_by_mode)

# Learned policy distribution
policy_dist = pd.Series(policy_actions).value_counts(normalize=True)

# Expected profit under learned policy
policy_profit = 0

for mode, pct in policy_dist.items():
    # Extract only the shipping mode from the composite key
    shipping_mode_name = mode.split('|')[1]
    policy_profit += pct * profit_by_mode[shipping_mode_name]

historical_profit = train_df["Order Profit Per Order"].mean()

print("Historical Profit:", historical_profit)
print("Policy Profit:", policy_profit)

improvement = (
    (policy_profit - historical_profit)
    / historical_profit
) * 100

print("Improvement %:", improvement)

Shipping Mode_str
First Class       4.435445
Same Day          4.193745
Second Class      4.298993
Standard Class    4.421882
Name: Order Profit Per Order, dtype: float64
Historical Profit: 4.385647162542691
Policy Profit: 4.334187491298916
Improvement %: -1.1733655110991565


In [130]:
print("Historical Profit:", historical_profit)
print("Policy Profit:", policy_profit)

Historical Profit: 4.385647162542691
Policy Profit: 4.334187491298916


In [132]:
historical_late = train_df["Late_delivery_risk"].mean()

late_by_mode = train_df.groupby(
    "Shipping Mode_str"
)["Late_delivery_risk"].mean()

policy_late = 0

for mode, pct in policy_dist.items():
    # Extract only the shipping mode from the composite key
    shipping_mode_name = mode.split('|')[1]
    policy_late += pct * late_by_mode[shipping_mode_name]

print("Historical Late Risk:", historical_late)
print("Policy Late Risk:", policy_late)

Historical Late Risk: 0.5183610359489756
Policy Late Risk: 0.6596767162893071


# Save Results

In [114]:
policy_df.to_csv(
    "q_learning_policy.csv",
    index=False
)

np.save(
    "q_table.npy",
    Q
)

print("Policy and Q-table saved")

Policy and Q-table saved


# **STEP 7: SARSA IMPLEMENTATION**


# **STEP 8: COMPARISON OF PROFIT, REWARD, AND ON-TIME DELIVERY PERFORMANCE**


* State = information known BEFORE shipment decision
* Action = shipping method / route / priority decision
* Reward = profit − lateness penalty
* Next state = next order environment

Reward Function

Rt=α(Profit)−β(Late Shipment Penalty)

Where:

α controls profit importance

β controls delivery performance importance

A more detailed version:

Rt=αPt−βLt−γCt

Where:
Pt = profit

Lt = lateness indicator or delay days

Ct = shipping cost

# STEP 8: Q-learning or SARSA environment setup